In [1]:
import requests
from bs4 import BeautifulSoup

import pandas as pd
import os
import warnings
warnings.filterwarnings("ignore")

from tqdm import tqdm

In [6]:
# dapatkan url berita
def getUrlFromPage(url, headers, keyword):
    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, "html.parser")

        titles = soup.find_all("h3", class_="media__title")

        data = [
            {
                "title": a.get_text(strip=True),
                "link": a["href"],
                "keyword": keyword
            }
            for a in soup.select("h3.media__title a.media__link")
        ]

        return data

    except Exception as e:
        print(f"Error on page {url}: {e}")
        return None

def main(keyword="tbc"):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        next_page = True
        page = 1
        all_url = []

        # ubah
        limit_page = 200
        start_page_url = f"https://www.detik.com/search/searchall?query={keyword}&result_type=relevansi&page="
        output_path = "data"
        filename = f"list_url_{keyword}.csv"

        while next_page:
            current_page_url = start_page_url+str(page)
            response = requests.get(current_page_url, headers=headers)

            print(f"page={page}, url='{current_page_url}', {response}")

            url = getUrlFromPage(current_page_url, headers, keyword)

            all_url.extend(url)

            if page == limit_page:
                next_page = False

            page += 1

        os.makedirs(output_path, exist_ok=True)

        df = pd.DataFrame(all_url)
        df.to_csv(f"{output_path}/{filename}", index=False)

    except Exception as e:
        print(f"Error: {e}")
        return None

main("tbc")
main("diabetes")

page=1, url='https://www.detik.com/search/searchall?query=tbc&result_type=relevansi&page=1', <Response [200]>
page=2, url='https://www.detik.com/search/searchall?query=tbc&result_type=relevansi&page=2', <Response [200]>
page=3, url='https://www.detik.com/search/searchall?query=tbc&result_type=relevansi&page=3', <Response [200]>
page=4, url='https://www.detik.com/search/searchall?query=tbc&result_type=relevansi&page=4', <Response [200]>
page=5, url='https://www.detik.com/search/searchall?query=tbc&result_type=relevansi&page=5', <Response [200]>
page=6, url='https://www.detik.com/search/searchall?query=tbc&result_type=relevansi&page=6', <Response [200]>
page=7, url='https://www.detik.com/search/searchall?query=tbc&result_type=relevansi&page=7', <Response [200]>
page=8, url='https://www.detik.com/search/searchall?query=tbc&result_type=relevansi&page=8', <Response [200]>
page=9, url='https://www.detik.com/search/searchall?query=tbc&result_type=relevansi&page=9', <Response [200]>
page=10, u

In [7]:
# merge
df_diabetes = pd.read_csv("data/list_url_diabetes.csv")
df_tbc = pd.read_csv("data/list_url_tbc.csv")

df = pd.concat([df_diabetes, df_tbc], ignore_index=True)
df.to_csv("data/list_url.csv")

In [8]:
# seleksi non artikel
df = pd.read_csv("data/list_url.csv")

def is_article(url):
    return ("berita" in url
            and "detiktv" not in url
            and "video" not in url
            and "20.detik.com" not in url)

df = df[df["link"].apply(is_article)]
df.to_csv("data/list_url_selected.csv")

In [9]:
# df.rename(columns={'label': 'keyword'}, inplace=True)
df

,Unnamed: 0,title,link,keyword
0,0,Andre Rosiade Bantu Istri Pemulung Pengidap Di...,https://news.detik.com/berita/d-8452634/andre-...,diabetes
1,1,Pasien Diabetes Tak Lagi Perlu Minum Obat Seum...,https://health.detik.com/berita-detikhealth/d-...,diabetes
8,8,18 April Hari Diabetes Nasional dan Kenali Car...,https://www.detik.com/bali/berita/d-8450035/18...,diabetes
14,14,Terungkap Pemicu Terbesar Gagal Ginjal Makin M...,https://health.detik.com/berita-detikhealth/d-...,diabetes
23,23,Jebakan 'Hidden Sugar' di Minuman Manis yang B...,https://health.detik.com/berita-detikhealth/d-...,diabetes
...,...,...,...,...
3830,3830,Pemkot Tangerang Raih 48 Penghargaan Sepanjang...,https://news.detik.com/berita/d-6478020/pemkot...,tbc
3831,3831,Kata Dokter Paru soal 619 Anak di Bantul Kena ...,https://health.detik.com/berita-detikhealth/d-...,tbc
3832,3832,1.332 Warga Tulungagung Positif TBC,https://www.detik.com/jatim/berita/d-6477178/1...,tbc
3836,3836,Jepang yang Bakal Wajibkan WNI Tes TBC Mulai 2...,https://health.detik.com/berita-detikhealth/d-...,tbc


In [10]:
df = pd.read_csv("data/list_url_selected.csv")
df = df.rename(columns={'label': 'keyword'})

In [11]:
headers = {"User-Agent": "Mozilla/5.0"}

def get_content(url):
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.text, "html.parser")

        container = soup.select_one("div.detail__body-text.itp_bodycontent")

        if not container:
            container = soup.select_one("div.detail__body-text")

        if not container:
            container = soup.select_one("div.itp_bodycontent")

        if not container:
            print("Failed selector:", url)
            return None

        paragraphs = container.find_all("p")

        texts = []
        for p in paragraphs:
            txt = p.get_text(strip=True)

            if not txt:
                continue
            if "ADVERTISEMENT" in txt.upper():
                continue

            texts.append(txt)

        return " ".join(texts) if texts else None

    except:
        return None

tqdm.pandas()

content_series = df["link"].progress_apply(get_content)

100%|██████████| 2519/2519 [39:39<00:00,  1.06it/s]  


In [12]:
df['content'] = content_series

In [13]:
df

,Unnamed: 0.1,Unnamed: 0,title,link,keyword,content
0,0,0,Andre Rosiade Bantu Istri Pemulung Pengidap Di...,https://news.detik.com/berita/d-8452634/andre-...,diabetes,Wakil Ketua Komisi VI DPR RI dari Fraksi Gerin...
1,1,1,Pasien Diabetes Tak Lagi Perlu Minum Obat Seum...,https://health.detik.com/berita-detikhealth/d-...,diabetes,Ketua Umum Perkumpulan Endokrinologi Indonesia...
2,8,8,18 April Hari Diabetes Nasional dan Kenali Car...,https://www.detik.com/bali/berita/d-8450035/18...,diabetes,"Setiap tahun, tanggal 18 April selalu dipering..."
3,14,14,Terungkap Pemicu Terbesar Gagal Ginjal Makin M...,https://health.detik.com/berita-detikhealth/d-...,diabetes,Kementerian Kesehatan RI mengungkap tren mengk...
4,23,23,Jebakan 'Hidden Sugar' di Minuman Manis yang B...,https://health.detik.com/berita-detikhealth/d-...,diabetes,Kabar baik bagi kesehatan publik di Indonesia....
...,...,...,...,...,...,...
2514,3830,3830,Pemkot Tangerang Raih 48 Penghargaan Sepanjang...,https://news.detik.com/berita/d-6478020/pemkot...,tbc,"Sepanjang tahun 2022, Pemerintah Kota (Pemkot)..."
2515,3831,3831,Kata Dokter Paru soal 619 Anak di Bantul Kena ...,https://health.detik.com/berita-detikhealth/d-...,tbc,Dinas Kesehatan Kabupaten Bantul melaporkan ad...
2516,3832,3832,1.332 Warga Tulungagung Positif TBC,https://www.detik.com/jatim/berita/d-6477178/1...,tbc,Dinas Kesehatan Tulungagung menemukan 12.384 w...
2517,3836,3836,Jepang yang Bakal Wajibkan WNI Tes TBC Mulai 2...,https://health.detik.com/berita-detikhealth/d-...,tbc,Kementerian Kesehatan RI (Kemenkes) menanggapi...


In [14]:
df.isnull().sum()

Unnamed: 0.1     0
Unnamed: 0       0
title            0
link             0
keyword          0
content         22
dtype: int64

In [15]:
df.to_csv('data/dataset.csv')